In [12]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

np.random.seed(42)

In [13]:
# Load the train and test data
data_dir = Path.cwd().parent /"data"/"raw"
train = pd.read_csv(data_dir/"train.csv")
test = pd.read_csv(data_dir/"test.csv")

In [14]:

def add_engineered_features(df):
    df = df.copy()
    df["log_var"] = np.log1p(df["var"])
    df["log_diff_var"] = np.log1p(df["diff_var"])
    df["log_diff2_var"] = np.log1p(df["diff2_var"])
    df["peak_ratio_smooth10"] = df["smooth10_n_peaks"] / (df["n_peaks"] + 1)
    df["peak_ratio_smooth20"] = df["smooth20_n_peaks"] / (df["n_peaks"] + 1)
    df["diff_peak_ratio"] = df["diff_peaks"] / (df["n_peaks"] + 1)
    return df

In [15]:

def cv_f1_for_features(train_df, feat_cols):
    for df in [train_df]:
        df["channel"] = df["channel"].astype("category")
    X = train_df[feat_cols]
    y = train_df["anomaly"]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(train_df))
    for tr_idx, val_idx in skf.split(X, y):
        model = lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.03, num_leaves=15,
            min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
            class_weight="balanced", random_state=42, verbosity=-1,
        )
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx], categorical_feature=["channel"])
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    best_f1 = max(f1_score(y, (oof > t).astype(int)) for t in np.arange(0.1, 0.9, 0.01))
    return best_f1

In [16]:
# --- raw features only ---
raw_feat_cols = [c for c in train.columns if c not in ["id", "anomaly"]]
f1_raw = cv_f1_for_features(train.copy(), raw_feat_cols)
print(f"Raw features only:              best CV F1 = {f1_raw:.4f}")

# --- raw + engineered features ---
train_fe = add_engineered_features(train)
fe_feat_cols = [c for c in train_fe.columns if c not in ["id", "anomaly"]]
f1_fe = cv_f1_for_features(train_fe, fe_feat_cols)
print(f"Raw + engineered features:      best CV F1 = {f1_fe:.4f}")

print()
if f1_fe > f1_raw:
    print("Decision: KEEP engineered features, they improved F1.")
else:
    print("Decision: DROP engineered features, no improvement. Use raw feature set.")


Raw features only:              best CV F1 = 0.9121
Raw + engineered features:      best CV F1 = 0.9094

Decision: DROP engineered features, no improvement. Use raw feature set.
